In [2]:
sam1=SAM()
sam1.load_data('Active_SAM_joined/SAM_DR_ncbi_03142025.h5ad')

In [1]:
!pip install anndata==0.8.0

In [2]:
!pip install loompy

In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import loompy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sam1.adata.obs

,n_genes,n_counts,key
AAACCTGAGAGTTGGC 97,757,1220.0,97
AAACCTGCAAGCGTAG 97,979,1744.0,97
AAACCTGCAGTAGAGC 97,1236,1836.0,97
AAACCTGCATCACCCT 97,865,1473.0,97
AAACCTGGTAAGAGAG 97,574,1125.0,97
...,...,...,...
TTTGTCAGTTGAACTC 96,2473,5308.0,96
TTTGTCATCAGCTGGC 96,1042,1480.0,96
TTTGTCATCATCATTC 96,1440,2276.0,96
TTTGTCATCCGATATG 96,1338,2109.0,96


In [4]:
sam1.adata.obs = sam1.adata.obs.drop(['agrp_label','leiden_clusters','subclass_id_label_mapping','subclass_id_label_mapping_nounlabeled','agrp_labeled','subclass_id_label_reduced_mapping','subclass_id_label_reduced_mapping_nounlabeled','eq_subclass','eq_subclass_fraction','eq_subclass_nounlabeled'], axis = 1)

In [5]:
sam1.adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,n_genes,n_counts,key,nCount_SCT,nFeature_SCT,hicat_low,hicat_high,hicat_merged,subclass_id_label_lc,subclass_id_label_reduced_lc
AAACCTGAGACCCACC,zebrafish,2593,868,868,2593,72,2186,868,11,70,70,160,160
AAACCTGAGCTAAGAT,zebrafish,1271,753,753,1271,72,1615,753,13,120,120,49,49
AAACCTGAGGGTCGAT,zebrafish,4150,1897,1901,4154,72,2477,1662,14,113,110,45,45
AAACCTGAGTGAAGTT,zebrafish,1201,799,799,1201,72,1575,799,14,285,197,396,396
AAACCTGCACACCGAC,zebrafish,2128,1232,1233,2129,72,2079,1232,14,312,197,191,191
...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTGTCGCGGTT,zebrafish,1481,907,907,1481,97,1878,906,14,330,197,111,111
TTTGGTTTCACCCGAG,zebrafish,3181,1783,1783,3181,97,2697,1783,14,310,108,76,76
TTTGTCAGTATGAATG,zebrafish,1051,627,627,1051,97,1983,669,14,113,110,174,174
TTTGTCATCCCTAACC,zebrafish,2901,1679,1681,2903,97,2633,1679,14,309,197,526,526


In [6]:
sam1.adata.obs = sam1.adata.obs.drop(['subclass_id_label_lc','subclass_id_label_reduced_lc'],axis = 1)

In [7]:
sam1.adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,n_genes,n_counts,key,nCount_SCT,nFeature_SCT,hicat_low,hicat_high,hicat_merged
AAACCTGAGACCCACC,zebrafish,2593,868,868,2593,72,2186,868,11,70,70
AAACCTGAGCTAAGAT,zebrafish,1271,753,753,1271,72,1615,753,13,120,120
AAACCTGAGGGTCGAT,zebrafish,4150,1897,1901,4154,72,2477,1662,14,113,110
AAACCTGAGTGAAGTT,zebrafish,1201,799,799,1201,72,1575,799,14,285,197
AAACCTGCACACCGAC,zebrafish,2128,1232,1233,2129,72,2079,1232,14,312,197
...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTGTCGCGGTT,zebrafish,1481,907,907,1481,97,1878,906,14,330,197
TTTGGTTTCACCCGAG,zebrafish,3181,1783,1783,3181,97,2697,1783,14,310,108
TTTGTCAGTATGAATG,zebrafish,1051,627,627,1051,97,1983,669,14,113,110
TTTGTCATCCCTAACC,zebrafish,2901,1679,1681,2903,97,2633,1679,14,309,197


In [ ]:
for i in range(1):
    dat = sc.read_loom('Subclustering/subset_Allen_institute_Full_cluster_30_'+str(i)+'.loom')
    dat.obs_names = dat.obs['obs_names']
    dat.var_names = list(dat.var['x'])
    
    sam=SAM(dat)
    sam.preprocess_data()
    sam.run()
    
    sam.adata.obs_names_make_unique()
    sam.adata.var_names_make_unique()

    sam1.adata.obs_names_make_unique()
    sam1.adata.var_names_make_unique()
    
    sams = {'mg':sam,'dr':sam1}

    sm = SAMAP(
        sams,
        f_maps = 'BLASTMAPPING/maps/active_maps/',
    )
    
    sm.run(pairwise=True)
    
    save_samap(sm , 'Active_SAMap_Joined/sm_Allen_Full_dr_03122025_ncbi_cluster_30_' + str(i) + '.pkl')

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8278407496988738
Computing the UMAP embedding...
Elapsed time: 1178.8732674121857 seconds
Not updating the manifold...
